# Self-consistency: choices

**Difficulty:** Beginner

Same idea as the Noul version, applied to `Choice`: run the same moderation rubric against the same post more than once, and check whether the *picked label* — not just the probability — holds still.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/consistency_choice_cookbook (click to expand)</summary>

# Self-consistency: choices

> Add an uncertain outcome to moderation decisions and compare label agreement with the share of automatic actions.

This cookbook takes one borderline user post, runs an 8-question moderation rubric over it 15 times, and checks whether each answer holds still across the repeats. Every check is a `Choice`, so each answer is one label from a fixed set. In a moderation pipeline that label is the routing decision: remove or leave up, escalate or auto-resolve, send to the threat, spam, or general queue. When the label wobbles from one run to the next, the same post routes to different places for no good reason.

What to look for: picked labels can flip inside a single condition, including TypeSafe, and conditions disagree with each other. In this cookbook's run, TypeSafe's raw label agreement across repeats is 90.8% — lower than some LLM conditions (Haiku at temperature 0 hit 100%) but with a mean probability standard deviation of `0.0098`, about 2.5x to 5.6x lower than five of the six LLM probability conditions tested.

## The state: a borderline user post

The post is built to sit on the fence: heated, insulting language aimed partly at a person and partly at the argument and community, an off-platform invite (a link to another site), one prior account strike, and four user reports. There is no single obvious answer — small wording differences shouldn't randomly move the same post between enforcement paths.

```json
{
  "post_id": "P-88213",
  "author": { "user_id": "u/4471", "account_age_days": 38, "prior_strikes": 1, "followers": 210 },
  "context": {
    "surface": "public reply",
    "in_reply_to": "another user defending a game patch",
    "community": "r/gamedebates"
  },
  "content": {
    "text": "Are you seriously this dense? Anyone who defends that patch is a complete clown and should be embarrassed to even post here. People like you are what's ruining this community and honestly you need to be dealt with. Come say it to my face, invite's right here. Keep it up and I'll end your whole channel.",
    "has_link": true,
    "link_domain": "discord.gg",
    "language": "en"
  },
  "reports": { "user_reports": 4, "report_reasons": ["harassment", "spam", "threat"] }
}
```

## The rubric: 8 `Choice` questions

Each question has a fixed, mutually-exclusive label set: `category` (policy violation type), `primary_risk`, `target`, `action` (allow/warn/remove/strike/escalate), `queue` (which moderation queue owns it), `link_handling`, `review_path` (who makes the final call), and `severity`.

## Probability std dev (mean, across all 8 questions x 15 repeats)

```
condition                    mean prob std   x TypeSafe
claude-haiku-4-5 t=0               0.0012        0.12x
claude-haiku-4-5 t=default         0.0516        5.29x
gpt-5.4-mini t=0                   0.0312        3.20x
gpt-5.4-mini t=default             0.0543        5.56x
gpt-5.5-reasoning                  0.0305        3.12x
claude-opus-4-8-reasoning          0.0245        2.52x
typesafe_choice                    0.0098        1.00x
```

Small changes can still switch the top label when two labels are close — a low probability std dev doesn't mean the picked label never moves, only that the underlying distribution is more stable.

## Let uncertain probabilities produce an uncertain decision

Return `uncertain` when the top probability is below `0.60`, and send that case to a human — using the returned `probabilities`, not the separate `confidence` field, and no extra model call. Applying that policy to TypeSafe's 15 repeats:

```
condition          raw agree   policy agree   uncertain   automatic   conflicts
typesafe_choice       90.8%         99.2%        25.8%       74.2%          0
```

Agreement rose from 90.8% (forcing every repeat to a concrete label) to 99.2% once low-confidence repeats are allowed to say "uncertain" instead. 25.8% of answers came back uncertain, 74.2% automatic. `primary_risk` and `link_handling` came back uncertain on every repeat; `category` alternated between `Violence` and `uncertain`, crossing the `0.60` threshold on some repeats and not others — but **no question ever produced two different concrete labels** across the 15 repeats. The threshold changes whether TypeSafe commits to an answer, not which concrete answer it commits to.

None of this shows accuracy or superiority — Haiku at temperature 0 had 100% raw agreement on this same post, with no abstentions at all.

> The full cookbook runs this rubric against four other LLMs across multiple temperature/format settings, plots heatmaps of every sample, and ships a disk-backed cache — all trimmed here, since comparing vendors isn't this repo's purpose. Read the full cookbook at [docs.typesafe.ai/cookbooks/consistency_choice_cookbook](https://docs.typesafe.ai/cookbooks/consistency_choice_cookbook) any time.

</details>

## The core idea

Exactly the same self-consistency check as the Noul cookbook, but for `Choice`: the thing that can move across repeats isn't a single probability, it's *which label wins*. The cookbook's finding is subtle and worth holding onto precisely — TypeSafe's underlying probability distribution was more stable than most LLM conditions tested, but its raw label agreement (90.8%) was still lower than a couple of LLM conditions. Stability of the distribution and stability of the picked label aren't the same measurement, and a low-confidence repeat flipping to "uncertain" isn't the same failure as a repeat flipping to a *different concrete answer* — the cookbook found the second kind never happened at all in its run.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const post = {
  post_id: "P-88213",
  author: { user_id: "u/4471", account_age_days: 38, prior_strikes: 1, followers: 210 },
  context: {
    surface: "public reply",
    in_reply_to: "another user defending a game patch",
    community: "r/gamedebates",
  },
  content: {
    text:
      "Are you seriously this dense? Anyone who defends that patch is a complete clown and should be embarrassed to even post here. People like you are what's ruining this community and honestly you need to be dealt with. Come say it to my face, invite's right here. Keep it up and I'll end your whole channel.",
    has_link: true,
    link_domain: "discord.gg",
    language: "en",
  },
  reports: { user_reports: 4, report_reasons: ["harassment", "spam", "threat"] },
};

const NUM_SAMPLES = 5;
const picks: string[] = [];

for (let i = 0; i < NUM_SAMPLES; i++) {
  const response = await client.systemOne({
    state: post,
    questions: {
      category: choice(
        "What is the single most applicable content-policy category for this post?",
        {
          None: "No policy violation of any kind.",
          Harass: "Insults or demeans a person, with no threat of harm and no protected-class attack.",
          Hate: "Attacks a person or group over a protected characteristic (race, religion, gender, ...).",
          Violence: "Makes a credible threat of harm or incites violence against someone.",
          Spam: "Unsolicited promotion or link spam, with no personal attack.",
          Sexual: "Sexual or adult content.",
        },
      ),
    },
  });
  const { category } = response.answers;
  picks.push(`${category.choice} (${category.confidence.toFixed(2)})`);
}

console.log("category, 5 repeated calls:", picks);

## What just happened

Five independent calls, same post, same `category` question — this is deliberately the cookbook's own most-contested question (it alternated between `Violence` and low-confidence uncertainty in their run). Watch for two different things in your own output: does `category.choice` ever change between repeats, and does `category.confidence` swing enough that a `0.60` cutoff would flip some repeats to "uncertain"?

**Try it yourself:** add the `severity` question (`None` / `Low` / `Medium` / `High`) to the same call and run it 5 times too. Predict whether it's more or less stable than `category` — the post's threat-like wording is deliberately ambiguous, so severity might wobble even more.

## Related

- `cookbooks/self-consistency/nouls.ipynb` — the same self-consistency check, for `Noul` instead of `Choice`
- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the `0.60` uncertain-threshold pattern here is the same idea as confidence-gated routing, applied to raw `probabilities` instead of the `confidence` field
- `lessons/core-concepts/0002-choice.ipynb` — the `Choice` primitive itself

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.